In [25]:
# This notebook creates a single 80/20 train-test split from the master dataset
import numpy as np
import pandas as pd

df_master = pd.read_parquet("PAKISTAN/DATA/4.PK_MASTER_DATASET.parquet", engine="pyarrow")

In [26]:
print("\nDtypes:")
print(df_master.dtypes)


Dtypes:
hhidCaseIdentification       string[python]
hv000CountryCode                   category
hv001ClusterNumber                    int32
hv002HouseholdNumber                  int32
hv005SimplilingWeight                 int32
hv270WealthIndex                       int8
hv206HasElectricity                 float64
hv226CookingFuel                    float64
hv241FoodCookedInHouse              float64
hv242SeparateKitchenYesNo           float64
hv209HasRefrigerator                float64
hv221HasLandLine                    float64
hv243aHasMobilePhone                float64
hv208HasTelevision                  float64
hv024RegionDivision                    int8
hv025TypeOfResidence                   int8
hv219SexOfHead                         int8
hv220AgeOfHead                         int8
hv106_01EducationOfHead            category
hv115_01MaritalStatus              category
hv009FamilySize                        int8
hv247HasBankAccount                category
hv216HouseSize         

In [27]:
print("\nMissing values:")
print(df_master.isna().sum().sort_values(ascending=False).head())


Missing values:
hv242SeparateKitchenYesNo    1189
hv115_01MaritalStatus           0
hv247HasBankAccount             0
hv216HouseSize                  0
hv014NoOfChildren               0
dtype: int64


In [28]:
drop_cols_mepi_other = [
    "hv000CountryCode",
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",
    "hv005SimplilingWeight",
    "weight",
    "hv206HasElectricity",
    "hv226CookingFuel",
    "hv241FoodCookedInHouse",
    "hv242SeparateKitchenYesNo",
    "hv209HasRefrigerator",
    "hv221HasLandLine",
    "hv243aHasMobilePhone",
    "hv208HasTelevision",
    "Depr_Elec",
    "Depr_CleanFuel",
    "Depr_Refrigerator",
    "Depr_TV",
    "Depr_Phone",
    "Depr_Kitchen",
    "MEPI_score",
]

In [29]:
df_master = df_master.drop(columns=drop_cols_mepi_other)

In [30]:
print("\nDtypes:")
print(df_master.dtypes)

print("\nMissing values:")
print(df_master.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex               int8
hv024RegionDivision            int8
hv025TypeOfResidence           int8
hv219SexOfHead                 int8
hv220AgeOfHead                 int8
hv106_01EducationOfHead    category
hv115_01MaritalStatus      category
hv009FamilySize                int8
hv247HasBankAccount        category
hv216HouseSize                 int8
hv014NoOfChildren              int8
v714CurrentlyWorking        float64
v717Occupation              float64
745aHouseOwnership          float64
EnergyPoor                     int8
dtype: object

Missing values:
hv270WealthIndex        0
hv024RegionDivision     0
hv025TypeOfResidence    0
hv219SexOfHead          0
hv220AgeOfHead          0
dtype: int64


In [ ]:
#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]

for col in cols_to_numeric:
    df_master[col] = pd.to_numeric(
        df_master[col],
        errors="coerce"
    )

In [31]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")

    df[[
        "EnergyPoor",
           ]] = df[[
        "EnergyPoor",
       ]].astype("int8")

    return df

In [32]:
df_ml = restore_ml_dtypes(df_master)

In [33]:
print("\nDtypes:")
print(df_ml.dtypes)

print("\nMissing values:")
print(df_ml.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex           category
hv024RegionDivision        category
hv025TypeOfResidence       category
hv219SexOfHead             category
hv220AgeOfHead                 int8
hv106_01EducationOfHead    category
hv115_01MaritalStatus      category
hv009FamilySize                int8
hv247HasBankAccount        category
hv216HouseSize                 int8
hv014NoOfChildren              int8
v714CurrentlyWorking       category
v717Occupation             category
745aHouseOwnership         category
EnergyPoor                     int8
dtype: object

Missing values:
hv270WealthIndex        0
hv024RegionDivision     0
hv025TypeOfResidence    0
hv219SexOfHead          0
hv220AgeOfHead          0
dtype: int64


In [34]:
from sklearn.model_selection import train_test_split

LABEL = "EnergyPoor"

# 1. Split
train_df, test_df = train_test_split(
    df_ml,
    test_size=0.2,
    random_state=42,
    stratify=df_ml[LABEL]
)

# 2. Safety check (no leakage)
assert set(train_df.index).isdisjoint(test_df.index), \
    "Train and test sets overlap!"

# 3. Optional sanity checks
print(train_df.shape, test_df.shape)
print(train_df[LABEL].value_counts(normalize=True))
print(test_df[LABEL].value_counts(normalize=True))

# 4. Save

train_df.to_parquet("PAKISTAN/DATA/6.PK_TRAIN_80SET.parquet",engine="pyarrow",compression="snappy", index=False
)

test_df.to_parquet("PAKISTAN/DATA/6.PK_TEST_20SET.parquet",engine="pyarrow",compression="snappy",index=False
)
print(f"✅ File saved.")

(11556, 15) (2890, 15)
EnergyPoor
0    0.573036
1    0.426964
Name: proportion, dtype: float64
EnergyPoor
0    0.57301
1    0.42699
Name: proportion, dtype: float64
✅ File saved.
